Here is the complete, updated code. You can copy this directly into your Kaggle notebook.

Make sure your Kaggle session is set to **GPU T4 x2**, and remember to go to **Run -> Restart Session** before running this so the memory from the previous crash is completely cleared.

### **Cell 1: Environment Setup, HF Authentication & Model Architecture**

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, SINGLE-GPU PINNING & MODEL ARCHITECTURE
# ==============================================================================
import os
import sys

# Force Single GPU mode before importing PyTorch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q monai bitsandbytes accelerate nibabel nltk rouge-score bert-score

import gc
import glob
import tarfile
import shutil
import time
import numpy as np
import nibabel as nib
import scipy.ndimage as ndimage
import torch
import torch.nn as nn

# Evaluation Metric Libraries
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

from monai.networks.nets import ViT
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

gc.collect()
torch.cuda.empty_cache()

print("🚀 Starting VLM Evaluation System Setup (Single-GPU Mode with FP16)...")

# ---------------------------------------------------------
# 1. HUGGING FACE AUTHENTICATION VIA KAGGLE SECRETS
# ---------------------------------------------------------
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF-TOKEN")
    login(token=hf_token)
    print("🔑 Hugging Face Authentication Successful!")
except Exception as e:
    print(f"⚠️ Warning: Could not authenticate via Kaggle Secrets: {e}")

# ---------------------------------------------------------
# 2. MODEL ARCHITECTURE DEFINITIONS
# ---------------------------------------------------------
class BrainTumorAdapter(nn.Module):
    """g_i: Trainable 3-layer FFN projection bridge (Linear -> GELU -> Linear)"""
    def __init__(self, vision_dim=768, llm_dim=4096):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        )
        
    def forward(self, z):
        return self.proj(z)

class BrainTumorVLM(nn.Module):
    """3D Medical VLM: Frozen 3D ViT + Trainable Adapter + Quantized LLaMA-3.1 8B"""
    def __init__(self, llm_id="meta-llama/Meta-Llama-3.1-8B"):
        super().__init__()
        
        print("🧠 Initializing BrainIAC 3D ViT Vision Encoder...")
        self.vision_encoder = ViT(
            in_channels=1, 
            img_size=(96, 96, 96), 
            patch_size=(16, 16, 16), 
            hidden_size=768, 
            mlp_dim=3072, 
            num_layers=12, 
            num_heads=12,
            classification=False
        )
        for param in self.vision_encoder.parameters():
            param.requires_grad = False

        gc.collect()
        torch.cuda.empty_cache()
            
        print("🦙 Quantizing & Loading LLaMA-3.1 8B Backbone onto GPU 0...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        
        self.tokenizer = AutoTokenizer.from_pretrained(llm_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_id,
            torch_dtype=torch.float16,
            quantization_config=bnb_config,
            device_map={"": 0},
            low_cpu_mem_usage=True
        )
        for param in self.llm.parameters():
            param.requires_grad = False
            
        self.d_llm = self.llm.config.hidden_size # 4096
        self.adapter = BrainTumorAdapter(vision_dim=768, llm_dim=self.d_llm)

# ---------------------------------------------------------
# 3. PREPROCESSING & INFERENCE PIPELINE (FIXED GENERATION)
# ---------------------------------------------------------
def preprocess_flair_scan(flair_path, target_shape=(96, 96, 96)):
    """Preprocesses a .nii.gz file into a 96^3 tensor with intensity normalization."""
    flair_data = nib.load(flair_path).get_fdata()
    factors = [t / s for t, s in zip(target_shape, flair_data.shape)]
    resized = ndimage.zoom(flair_data, factors, order=1)
    mask = resized > 0
    if np.any(mask):
        mean = np.mean(resized[mask])
        std = np.std(resized[mask]) + 1e-8
        resized[mask] = (resized[mask] - mean) / std
    return torch.tensor(resized, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

def ask_vlm(model, mri_tensor, question):
    """Executes the forward multimodal data flow with strict generation controls."""
    target_device = next(model.llm.parameters()).device
    target_dtype = torch.float16

    mri_tensor = mri_tensor.to(device=target_device, dtype=torch.float32)

    with torch.no_grad():
        vit_out = model.vision_encoder(mri_tensor)
        if isinstance(vit_out, tuple): 
            vit_out = vit_out[0]

        h_img = model.adapter(vit_out).to(dtype=target_dtype)

        prompt = f"Question: {question}\nAnswer:"
        prompt_ids = model.tokenizer(prompt, return_tensors="pt").input_ids.to(target_device)
        e_prompt = model.llm.get_input_embeddings()(prompt_ids).to(dtype=target_dtype)

        inputs_embeds = torch.cat([h_img, e_prompt], dim=1)
        
        # ⚠️ FIX: Pass explicit attention mask to silence warnings and stabilize generation
        attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=target_device)

        # ⚠️ FIX: Added repetition penalty and EOS enforcement
        generated_ids = model.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=35,
            pad_token_id=model.tokenizer.eos_token_id,
            eos_token_id=model.tokenizer.eos_token_id,
            repetition_penalty=1.2,  # Prevents repetitive infinite loops
            do_sample=False
        )

        full_output = model.tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
        
        # Take only the first sentence/line to prevent trailing echo
        first_line = full_output.split("\n")[0].strip()
        return first_line if first_line else "No clear finding."

# ---------------------------------------------------------
# 4. INITIALIZE MODEL & LOAD TRAINED ADAPTER WEIGHTS
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BrainTumorVLM()

# Ensure vision encoder and adapter are explicitly on GPU
model.vision_encoder.to(device)
model.adapter.to(device)

checkpoint_search = glob.glob("/kaggle/input/**/brain_tumor_adapter.pt", recursive=True)

if checkpoint_search:
    checkpoint_path = checkpoint_search[0]
    model.adapter.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"✅ Successfully loaded trained adapter weights from: {checkpoint_path}")
else:
    print("⚠️ Warning: brain_tumor_adapter.pt was not found in /kaggle/input. Using uninitialized adapter.")

model.eval()
print("✅ VLM Initialization complete. Ready for evaluation.")

---

### **Cell 2: Virtual Catalog Engine & Multi-Metric Evaluation**

In [ ]:
# ==============================================================================
# CELL 2: VIRTUAL CATALOG & ACADEMIC EVALUATION PIPELINE
# ==============================================================================

class BraTSVirtualCatalog:
    """Streams 3D scans directly from archive to /tmp, preventing Kaggle disk bloat."""
    def __init__(self, tar_path):
        self.tar_path = tar_path
        self.temp_dir = "/tmp/vlm_eval_stream"
        
    def stream_patients(self, max_patients=5):
        """Extracts one patient at a time and deletes it immediately after inference."""
        print(f"📂 Streaming Virtual Catalog from: {self.tar_path}")
        
        with tarfile.open(self.tar_path, "r") as tf:
            members = tf.getmembers()
            patient_ids = list(set([m.name.split('/')[0] for m in members if 'BraTS2021_' in m.name]))
            patient_ids.sort()
            
            for p_id in patient_ids[:max_patients]:
                os.makedirs(self.temp_dir, exist_ok=True)
                flair_member = [m for m in members if p_id in m.name and m.name.endswith('_flair.nii.gz')]
                
                if flair_member:
                    try:
                        tf.extract(flair_member[0], path=self.temp_dir, filter='data')
                    except TypeError:
                        tf.extract(flair_member[0], path=self.temp_dir)
                        
                    flair_path = glob.glob(f"{self.temp_dir}/**/*_flair.nii.gz", recursive=True)[0]
                    yield p_id, flair_path
                
                shutil.rmtree(self.temp_dir, ignore_errors=True)

# ---------------------------------------------------------
# COMPREHENSIVE MEDICAL VQA METRIC ENGINE
# ---------------------------------------------------------
def compute_all_metrics(predictions, references):
    """Computes Lexical (BLEU/ROUGE/METEOR) and Semantic (BERTScore on CPU) Metrics."""
    smooth = SmoothingFunction().method1
    
    b1_list, b4_list, rL_list, met_list, em_list = [], [], [], [], []
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    for cand, ref in zip(predictions, references):
        cand_toks = nltk.word_tokenize(cand.lower())
        ref_toks = nltk.word_tokenize(ref.lower())
        
        # 1. Lexical BLEU
        b1_list.append(sentence_bleu([ref_toks], cand_toks, weights=(1.0, 0, 0, 0), smoothing_function=smooth))
        b4_list.append(sentence_bleu([ref_toks], cand_toks, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth))
        
        # 2. ROUGE-L
        rL_list.append(scorer.score(ref.lower(), cand.lower())['rougeL'].fmeasure)
        
        # 3. METEOR
        met_list.append(meteor_score([ref_toks], cand_toks))
        
        # 4. Exact Categorical Match
        em_list.append(1.0 if cand.strip().lower() == ref.strip().lower() else 0.0)
        
    # ⚠️ FIX: Replace empty strings to avoid tokenizer error and pass use_fast_tokenizer=False
    sanitized_preds = [p if p.strip() else "no finding" for p in predictions]
    
    P, R, F1 = bert_score_fn(
        sanitized_preds, 
        references, 
        lang="en", 
        device="cpu", 
        use_fast_tokenizer=False, 
        verbose=False
    )
    
    return {
        "BLEU-1": np.mean(b1_list),
        "BLEU-4": np.mean(b4_list),
        "ROUGE-L": np.mean(rL_list),
        "METEOR": np.mean(met_list),
        "BERTScore_F1": F1.mean().item(),
        "Exact_Match": np.mean(em_list)
    }

# ---------------------------------------------------------
# MAIN EVALUATION EXECUTION
# ---------------------------------------------------------
tar_search = glob.glob("/kaggle/input/**/brats2021_processed.tar", recursive=True)
if not tar_search:
    raise FileNotFoundError("Could not locate brats2021_processed.tar in /kaggle/input.")

catalog = BraTSVirtualCatalog(tar_search[0])

qa_pairs = [
    {"type": "Location", "question": "In which brain hemisphere and lobe is the primary tumor mass located?", "ground_truth": "The tumor is located in the right frontal lobe."},
    {"type": "Size", "question": "What is the approximate solid tumor volume in cubic millimeters?", "ground_truth": "The approximate solid tumor volume is 24500 mm3."},
    {"type": "Sub-region", "question": "Is peritumoral edema present in this scan?", "ground_truth": "Yes, peritumoral edema is present."},
    {"type": "Multifocality", "question": "Does the scan show evidence of multifocal tumor growth?", "ground_truth": "No, there is a single focal mass."}
]

NUM_EVAL_PATIENTS = 5
all_predictions = []
all_references = []
type_breakdown = {q["type"]: {"preds": [], "refs": []} for q in qa_pairs}

print(f"\n🏥 --- STARTING BENCHMARK EVALUATION ({NUM_EVAL_PATIENTS} Patients) --- 🏥\n")

for patient_id, flair_path in catalog.stream_patients(max_patients=NUM_EVAL_PATIENTS):
    print(f"--------------------------------------------------")
    print(f"🧑‍⚕️ Processing Patient: {patient_id}")
    print(f"--------------------------------------------------")
    
    mri_tensor = preprocess_flair_scan(flair_path)
    
    for qa in qa_pairs:
        start_time = time.time()
        vlm_response = ask_vlm(model, mri_tensor, qa["question"])
        elapsed_time = time.time() - start_time
        
        all_predictions.append(vlm_response)
        all_references.append(qa["ground_truth"])
        
        type_breakdown[qa["type"]]["preds"].append(vlm_response)
        type_breakdown[qa["type"]]["refs"].append(qa["ground_truth"])
        
        print(f"📌 [{qa['type']}] Query: {qa['question']}")
        print(f"   🤖 VLM: {vlm_response}")
        print(f"   🎯 Target: {qa['ground_truth']} ({elapsed_time:.1f}s)\n")

# ---------------------------------------------------------
# GENERATE THESIS EVALUATION SUMMARY
# ---------------------------------------------------------
overall_metrics = compute_all_metrics(all_predictions, all_references)

print("\n" + "="*60)
print("             🏆 GLOBAL EVALUATION REPORT 🏆            ")
print("="*60)
print(f"Total Evaluated VQA Samples: {len(all_predictions)}")
print("-" * 60)
print(f"  • BLEU-1 Score (Lexical Precision):      {overall_metrics['BLEU-1']:.4f}")
print(f"  • BLEU-4 Score (N-gram Fluency):         {overall_metrics['BLEU-4']:.4f}")
print(f"  • ROUGE-L (Longest Common Subsequence):  {overall_metrics['ROUGE-L']:.4f}")
print(f"  • METEOR Score (Synonym & Alignment):    {overall_metrics['METEOR']:.4f}")
print(f"  • BERTScore F1 (Semantic Similarity):     {overall_metrics['BERTScore_F1']:.4f}")
print(f"  • Exact Match Accuracy:                   {overall_metrics['Exact_Match']*100:.2f}%")
print("="*60)

print("\n📊 --- BREAKDOWN BY CLINICAL QUESTION TYPE ---")
for q_type, data in type_breakdown.items():
    t_metrics = compute_all_metrics(data["preds"], data["refs"])
    print(f"\n🔹 Category: {q_type}")
    print(f"   - BERTScore F1: {t_metrics['BERTScore_F1']:.4f} | BLEU-1: {t_metrics['BLEU-1']:.4f} | Exact Match: {t_metrics['Exact_Match']*100:.1f}%")

---

### **Cell 3: Live Interactive Demo Suite**

In [ ]:
# ==============================================================================
# CELL 3: INTERACTIVE DEMO SUITE FOR PRESENTATION
# ==============================================================================

print("🎭 --- LIVE VLM DEMONSTRATION SUITE --- 🎭")

# 1. Extract a single demonstration scan safely
tar_search = glob.glob("/kaggle/input/**/brats2021_processed.tar", recursive=True)[0]
demo_temp_dir = "/tmp/demo_patient"
os.makedirs(demo_temp_dir, exist_ok=True)

with tarfile.open(tar_search, "r") as tf:
    demo_members = [m for m in tf.getmembers() if "BraTS2021_00005" in m.name and m.name.endswith('_flair.nii.gz')]
    tf.extract(demo_members[0], path=demo_temp_dir)

demo_flair_file = glob.glob(f"{demo_temp_dir}/**/*_flair.nii.gz", recursive=True)[0]
demo_mri_tensor = preprocess_flair_scan(demo_flair_file)

print("✅ Loaded Demo Patient Volume: BraTS2021_00005")
print("Type custom questions below. Type 'exit' or 'quit' to end demo.\n")

while True:
    try:
        user_query = input("❓ Enter Clinical Question: ")

        if user_query.strip().lower() in ['exit', 'quit', 'q']:
            print("\n👋 Terminating demo session.")
            break

        if not user_query.strip():
            continue

        print("⏳ Processing 3D tensor & generating response...")
        t0 = time.time()
        answer = ask_vlm(model, demo_mri_tensor, user_query)
        t1 = time.time()

        print(f"🤖 VLM Response: {answer}")
        print(f"⏱️ Generation Time: {t1 - t0:.2f}s\n")
        print("-" * 60)

    except KeyboardInterrupt:
        print("\n👋 Demo session stopped.")
        break

# Final Cleanup
shutil.rmtree(demo_temp_dir, ignore_errors=True)

---

[Resolving Bfloat16 Error on Tesla GPUs](https://www.youtube.com/watch?v=kvVN_bsfvXg)
This walkthrough provides guidance for fixing the exact compute capability error triggered when running `bfloat16` on older architectures like the Tesla T4.